# Chunking Strategies for RAG Systems

**Deliverable:** Theory + Python practical + retrieval evaluation

This notebook demonstrates how different chunking strategies affect RAG retrieval.

**Flow:** Document → Chunking → Embeddings → Vector Search → Evaluation

## 1. Theory — What is Chunking?

Chunking means breaking a large document into smaller pieces called **chunks** before creating embeddings.

Why? If we embed one huge document, the vector may represent many unrelated topics. Smaller, meaningful chunks usually make retrieval more focused.

### Chunk size
- Small chunks → better precision, but context can be lost.
- Large chunks → more context, but can contain irrelevant information.

### Chunk overlap
Overlap repeats some text between neighboring chunks. It protects information that lies near a chunk boundary.

A useful starting point is **10–20% overlap**, but the best value must be tested on the target data.

## 2. Chunking Strategies

| Strategy | Main idea | Good for |
|---|---|---|
| Fixed character | Split every N characters | Simple baseline |
| Token-based | Split every N tokens | Token-controlled systems |
| Sentence | Keep sentences together | Normal prose |
| Paragraph | Keep paragraphs together | Structured text |
| Markdown headers | Preserve headings and sections | Documentation |
| Recursive character | Try meaningful separators before smaller ones | General RAG |

### Small vs Large

**Small:** focused embeddings, but may lose context.

**Large:** more context, but may add noise.

There is no universally best chunk size. We should experiment and measure retrieval quality.

## 3. Setup

The practical below first uses standard Python so the basic idea is clear. Then it demonstrates LangChain's recursive splitter and an embedding/retrieval experiment.

If optional packages are missing, install them in the next cell.

In [19]:
# Uncomment and run if needed:
# %pip install -q langchain-text-splitters sentence-transformers scikit-learn pandas

## 4. Sample Airline Document

For the experiment, we use a small airline knowledge base. In a real project, replace this with your `.txt` document.

In [20]:
text = """
Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

Passengers can carry up to 7 kg of cabin baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

Cancellation Policy

Passengers can cancel a booking before departure.
Cancellation fees depend on the fare type and how close the cancellation is to departure.

Flight Change Policy

Passengers may request a flight date change subject to availability.
A change fee may apply depending on the fare conditions.
""".strip()

print(text)

Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

Passengers can carry up to 7 kg of cabin baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

Cancellation Policy

Passengers can cancel a booking before departure.
Cancellation fees depend on the fare type and how close the cancellation is to departure.

Flight Change Policy

Passengers may request a flight date change subject to availability.
A change fee may apply depending on the fare conditions.


## 5. Practical 1 — Fixed-Size Character Chunking

This is the simplest baseline. It cuts text into fixed character ranges.

**Trade-off:** simple and predictable, but it can cut through sentences.

In [21]:
def fixed_character_chunks(text, chunk_size=300, overlap=30):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = end - overlap

    return chunks

fixed_chunks = fixed_character_chunks(text, chunk_size=300, overlap=30)

for i, chunk in enumerate(fixed_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 1 ---
Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

Passengers can carry up to 7 kg of cabin baggage.
Checked baggage allo

--- Chunk 2 ---
 baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

Cancellation Policy

Passengers can cancel a booking before departure.
Cancellation fees depend on the fare type and how close the cancellation is to departure.

Flight Change Policy

Passenger

--- Chunk 3 ---
light Change Policy

Passengers may request a flight date change subject to availability.
A change fee may apply depending on the fare conditions.



## 6. Practical 2 — Sentence Chunking

Here we keep complete sentences together instead of cutting at an arbitrary character position.

In [22]:
import re

def sentence_chunks(text, sentences_per_chunk=3):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [
        " ".join(sentences[i:i + sentences_per_chunk])
        for i in range(0, len(sentences), sentences_per_chunk)
    ]

sentence_chunks_result = sentence_chunks(text, sentences_per_chunk=3)

for i, chunk in enumerate(sentence_chunks_result, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 1 ---
Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days. Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

--- Chunk 2 ---
Baggage Policy

Passengers can carry up to 7 kg of cabin baggage. Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

--- Chunk 3 ---
Cancellation Policy

Passengers can cancel a booking before departure. Cancellation fees depend on the fare type and how close the cancellation is to departure. Flight Change Policy

Passengers may request a flight date change subject to availability.

--- Chunk 4 ---
A change fee may apply depending on the fare conditions.



## 7. Practical 3 — Paragraph Chunking

Paragraphs often represent natural semantic units.

In [23]:
def paragraph_chunks(text):
    paragraphs = [p.strip() for p in text.split("\n\n")]
    return [p for p in paragraphs if p]

paragraph_chunks_result = paragraph_chunks(text)

for i, chunk in enumerate(paragraph_chunks_result, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 1 ---
Refund Policy

--- Chunk 2 ---
If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

--- Chunk 3 ---
Baggage Policy

--- Chunk 4 ---
Passengers can carry up to 7 kg of cabin baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

--- Chunk 5 ---
Cancellation Policy

--- Chunk 6 ---
Passengers can cancel a booking before departure.
Cancellation fees depend on the fare type and how close the cancellation is to departure.

--- Chunk 7 ---
Flight Change Policy

--- Chunk 8 ---
Passengers may request a flight date change subject to availability.
A change fee may apply depending on the fare conditions.



## 8. Practical 4 — Recursive Character Splitting

Recursive splitting tries larger meaningful separators first, then falls back to smaller separators if a piece is still too large.

Typical order:

**paragraph → line → sentence → word → character**

This usually gives more coherent chunks than blind fixed-size splitting.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(text)

for i, chunk in enumerate(recursive_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 1 ---
Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

--- Chunk 2 ---
Baggage Policy

Passengers can carry up to 7 kg of cabin baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

Cancellation Policy

--- Chunk 3 ---
Cancellation Policy

Passengers can cancel a booking before departure.
Cancellation fees depend on the fare type and how close the cancellation is to departure.

Flight Change Policy

--- Chunk 4 ---
Flight Change Policy

Passengers may request a flight date change subject to availability.
A change fee may apply depending on the fare conditions.



## 9. Practical 5 — Markdown Header Chunking

For Markdown documentation, headings carry important meaning. Header-aware splitting keeps the document structure available during retrieval.

Example structure:

```text
# Refund Policy
## Eligibility
## Processing Time
```

This is especially useful for manuals, policies and documentation.

In [25]:
markdown_text = """
# Refund Policy

Refunds are available according to fare rules.

## Eligibility

Eligible passengers can request a refund after cancellation.

## Processing Time

Eligible refunds are processed within 7 business days.

# Baggage Policy

## Cabin Baggage

Passengers can carry up to 7 kg of cabin baggage.
""".strip()

from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

markdown_chunks = markdown_splitter.split_text(markdown_text)

for i, chunk in enumerate(markdown_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

--- Chunk 1 ---
page_content='Refunds are available according to fare rules.' metadata={'Header 1': 'Refund Policy'}

--- Chunk 2 ---
page_content='Eligible passengers can request a refund after cancellation.' metadata={'Header 1': 'Refund Policy', 'Header 2': 'Eligibility'}

--- Chunk 3 ---
page_content='Eligible refunds are processed within 7 business days.' metadata={'Header 1': 'Refund Policy', 'Header 2': 'Processing Time'}

--- Chunk 4 ---
page_content='Passengers can carry up to 7 kg of cabin baggage.' metadata={'Header 1': 'Baggage Policy', 'Header 2': 'Cabin Baggage'}



## 10. Compare Chunk Counts

Before retrieval, compare how many chunks each method creates. More chunks are not automatically better.

In [26]:
chunk_sets = {
    "Fixed 300/30": fixed_chunks,
    "Sentence": sentence_chunks_result,
    "Paragraph": paragraph_chunks_result,
    "Recursive 300/30": recursive_chunks,
    "Markdown Header": [str(c) for c in markdown_chunks],
}

for name, chunks in chunk_sets.items():
    print(f"{name:22} -> {len(chunks)} chunks")

Fixed 300/30           -> 3 chunks
Sentence               -> 4 chunks
Paragraph              -> 8 chunks
Recursive 300/30       -> 4 chunks
Markdown Header        -> 4 chunks


## 11. Retrieval Experiment

Now we test whether different chunking configurations retrieve the information needed by the same baseline queries.

For a fair comparison:
- Keep the same embedding model.
- Keep the same queries.
- Keep the same Top-K.
- Change only the chunking configuration.

We use Sentence Transformers embeddings + cosine similarity for a local demonstration.

In [27]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

queries = [
    ("How long does an eligible refund take?", ["7 business days"]),
    ("How much cabin baggage can a passenger carry?", ["7 kg"]),
    ("Can a passenger cancel a booking before departure?", ["cancel", "before departure"]),
    ("Can a passenger change the flight date?", ["flight date change"]),
]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8140.75it/s]


In [28]:
def retrieve(query, chunks, top_k=3):
    chunk_embeddings = model.encode(chunks, normalize_embeddings=True)
    query_embedding = model.encode([query], normalize_embeddings=True)

    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [(int(i), float(scores[i]), chunks[i]) for i in top_indices]

def evaluate_chunking(chunks, queries, top_k=3):
    hits = 0
    rows = []

    for query, expected_keywords in queries:
        results = retrieve(query, chunks, top_k=top_k)
        retrieved_text = " ".join(r[2].lower() for r in results)

        hit = all(k.lower() in retrieved_text for k in expected_keywords)
        hits += int(hit)

        rows.append({
            "query": query,
            "hit": hit,
            "top_score": round(results[0][1], 4),
            "top_chunk": results[0][2]
        })

    hit_at_k = hits / len(queries)
    return hit_at_k, rows

## 12. Run the Evaluation

`Hit@3` asks: **Did the retrieved Top-3 context contain the expected information?**

Formula:

**Hit@3 = successful queries / total queries**

Higher is better for this simple baseline evaluation.

In [29]:
import pandas as pd

summary = []
details = {}

for name, chunks in chunk_sets.items():
    hit_at_3, rows = evaluate_chunking(chunks, queries, top_k=3)
    summary.append({
        "Configuration": name,
        "Chunk Count": len(chunks),
        "Hit@3": round(hit_at_3, 3)
    })
    details[name] = rows

results_df = pd.DataFrame(summary)
results_df

,Configuration,Chunk Count,Hit@3
0,Fixed 300/30,3,1.0
1,Sentence,4,1.0
2,Paragraph,8,1.0
3,Recursive 300/30,4,1.0
4,Markdown Header,4,0.5


## 13. Inspect Retrieval Results

Always inspect actual retrieved chunks, not only the metric. A high score can still hide irrelevant context.

In [30]:
for name, rows in details.items():
    print("\n" + "=" * 80)
    print(name)
    for row in rows:
        print(f"\nQuery: {row['query']}")
        print(f"Hit@3: {row['hit']} | Top score: {row['top_score']}")
        print(f"Top chunk: {row['top_chunk'][:250]}...")


Fixed 300/30

Query: How long does an eligible refund take?
Hit@3: True | Top score: 0.4538
Top chunk: Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

Passengers can carry...

Query: How much cabin baggage can a passenger carry?
Hit@3: True | Top score: 0.4094
Top chunk: Refund Policy

If a passenger cancels a flight, the eligible refund is processed within 7 business days.
Passengers should submit a refund request after cancellation. Refund eligibility depends on the fare rules.

Baggage Policy

Passengers can carry...

Query: Can a passenger cancel a booking before departure?
Hit@3: True | Top score: 0.5557
Top chunk:  baggage.
Checked baggage allowance depends on the fare type. Additional baggage may incur extra charges.

Cancellation Policy

Passengers can cancel a booking before departure.
Cancellatio

## 14. How to Interpret the Experiment

If a small chunk configuration has a lower score, it may be losing context.

If a large chunk configuration has a lower score, it may contain too much unrelated information.

If recursive splitting performs better, it suggests that preserving natural boundaries helped retrieval.

If Markdown/header splitting performs better on structured documentation, the document hierarchy is probably useful retrieval information.

The exact result depends on the dataset and queries.

### Additional metrics for a larger experiment

- **Recall@K:** how much relevant information was retrieved.
- **Precision@K:** how much of retrieved information was relevant.
- **MRR:** how high the first relevant result appears.
- **Answer accuracy:** whether the final LLM answer is correct using the retrieved context.

## 15. Best Practices

1. Start with a simple baseline.
2. Prefer semantic boundaries when the document has structure.
3. Use moderate overlap to protect chunk boundaries.
4. Avoid excessive overlap because it creates duplicate content.
5. Store metadata such as source, page, title and section.
6. Use the same evaluation queries for every configuration.
7. Keep embedding model, Top-K and retrieval settings constant during comparison.
8. Choose chunk size based on the document and question style.
9. Evaluate retrieval quality before optimizing the final LLM answer.
10. For production, test on real user-like queries.

### Practical starting point

For a general RAG system, start around **500–800 tokens** with roughly **10–20% overlap**, then benchmark smaller and larger configurations.

For structured policies/manuals, combine **section/header-aware splitting + recursive fallback**.

# Final Conclusion

Chunking directly affects RAG retrieval quality.

- **Small chunks:** focused but may lose context.
- **Large chunks:** contextual but may add noise.
- **Overlap:** reduces boundary-related information loss.
- **Recursive splitting:** balances document structure and chunk size.
- **Structure-aware splitting:** is especially useful for Markdown, manuals and policies.

The best chunking strategy is not chosen by theory alone. It should be selected by running the same baseline queries against multiple configurations and comparing retrieval results.